In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

In [ ]:
!pip install transformers datasets langchain accelerate langchain-community accelerate bitsandbytes

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

In [ ]:
!huggingface-cli login --token $hf_token --add-to-git-credential

In [ ]:
import re
import torch
import transformers
from transformers import AutoTokenizer
from langchain import LLMChain, HuggingFacePipeline, PromptTemplate
from datasets import Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0", split='test')
df = dataset.select(range(50)).to_pandas()

In [ ]:
test_sample = df.iloc[:50]

data = {"article": test_sample['article'].tolist()}
dataset = Dataset.from_dict(data)

### Pipelines

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain import LLMChain, HuggingFacePipeline, PromptTemplate

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
def summary_LOV(model_name):
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return model, tokenizer

In [ ]:
llama2 = "meta-llama/Llama-2-7b-chat-hf"
llama31 = "DISLab/SummLlama3.1-8B"
llama32 = "DISLab/SummLlama3.2-3B"

# model_llama2, tokenizer_llama2 = summary_LOV(llama2)
model_llama31, tokenizer_llama31 = summary_LOV(llama31)
# model_llama32, tokenizer_llama32 = summary_LOV(llama32)

In [ ]:
from transformers import pipeline

pipelines = {
    # "llama2": pipeline(
    #     "text-generation",
    #     model=model_llama2,
    #     tokenizer=tokenizer_llama2,
    #     device_map="auto",
    # ),
    "llama31": pipeline(
        "summarization", 
        model=model_llama31,
        tokenizer=tokenizer_llama31,
        device_map="auto",
    ),
    # "llama32": pipeline(
    #     "summarization", 
    #     model=model_llama32,
    #     tokenizer=tokenizer_llama32,
    #     device_map="auto",
    # )
}

### Prompting Strategy

In [ ]:
templates = {
    "zero": "Summarize the following article:\n\n{article}\n\nFinal Summary:",
    "few": """
                Here are some examples of summaries with respect to their articles:
                Article: "LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Details of how he'll mark his landmark birthday are under wraps. His agent and publicist had no comment on his plans. "I'll definitely have some sort of party," he said in an interview. "Hopefully none of you will be reading about it." Radcliffe's earnings from the first five Potter films have been held in a trust fund which he has not been able to touch. Despite his growing fame and riches, the actor says he is keeping his feet firmly on the ground. "People are always looking to say 'kid star goes off the rails,'" he told reporters last month. "But I try very hard not to go that way because it would be too easy for them." His latest outing as the boy wizard in "Harry Potter and the Order of the Phoenix" is breaking records on both sides of the Atlantic and he will reprise the role in the last two films. Watch I-Reporter give her review of Potter's latest » . There is life beyond Potter, however. The Londoner has filmed a TV movie called "My Boy Jack," about author Rudyard Kipling and his son, due for release later this year. He will also appear in "December Boys," an Australian film about four boys who escape an orphanage. Earlier this year, he made his stage debut playing a tortured teenager in Peter Shaffer's "Equus." Meanwhile, he is braced for even closer media scrutiny now that he's legally an adult: "I just think I'm going to be more sort of fair game," he told Reuters. E-mail to a friend . Copyright 2007 Reuters. All rights reserved.This material may not be published, broadcast, rewritten, or redistributed."
                Summary: "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor says he has no plans to fritter his cash away . Radcliffe's earnings from first five Potter films have been held in trust fund ."
                
                Article: "Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events. Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial. MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor." Here, inmates with the most severe mental illnesses are incarcerated until they're ready to appear in court. Most often, they face drug charges or charges of assaulting an officer --charges that Judge Steven Leifman says are usually "avoidable felonies." He says the arrests often result from confrontations with police. Mentally ill people often won't do what they're told when police arrive on the scene -- confrontation seems to exacerbate their illness and they become more paranoid, delusional, and less likely to follow directions, according to Leifman. So, they end up on the ninth floor severely mentally disturbed, but not getting any real help because they're in jail. We toured the jail with Leifman. He is well known in Miami as an advocate for justice and the mentally ill. Even though we were not exactly welcomed with open arms by the guards, we were given permission to shoot videotape and tour the floor. Go inside the 'forgotten floor' » . At first, it's hard to determine where the people are. The prisoners are wearing sleeveless robes. Imagine cutting holes for arms and feet in a heavy wool sleeping bag -- that's kind of what they look like. They're designed to keep the mentally ill patients from injuring themselves. That's also why they have no shoes, laces or mattresses. Leifman says about one-third of all people in Miami-Dade county jails are mentally ill. So, he says, the sheer volume is overwhelming the system, and the result is what we see on the ninth floor. Of course, it is a jail, so it's not supposed to be warm and comforting, but the lights glare, the cells are tiny and it's loud. We see two, sometimes three men -- sometimes in the robes, sometimes naked, lying or sitting in their cells. "I am the son of the president. You need to get me out of here!" one man shouts at me. He is absolutely serious, convinced that help is on the way -- if only he could reach the White House. Leifman tells me that these prisoner-patients will often circulate through the system, occasionally stabilizing in a mental hospital, only to return to jail to face their charges. It's brutally unjust, in his mind, and he has become a strong advocate for changing things in Miami. Over a meal later, we talk about how things got this way for mental patients. Leifman says 200 years ago people were considered "lunatics" and they were locked up in jails even if they had no charges against them. They were just considered unfit to be in society. Over the years, he says, there was some public outcry, and the mentally ill were moved out of jails and into hospitals. But Leifman says many of these mental hospitals were so horrible they were shut down. Where did the patients go? Nowhere. The streets. They became, in many cases, the homeless, he says. They never got treatment. Leifman says in 1955 there were more than half a million people in state mental hospitals, and today that number has been reduced 90 percent, and 40,000 to 50,000 people are in mental hospitals. The judge says he's working to change this. Starting in 2008, many inmates who would otherwise have been brought to the "forgotten floor" will instead be sent to a new mental health facility -- the first step on a journey toward long-term treatment, not just punishment. Leifman says it's not the complete answer, but it's a start. Leifman says the best part is that it's a win-win solution. The patients win, the families are relieved, and the state saves money by simply not cycling these prisoners through again and again. And, for Leifman, justice is served. E-mail to a friend ."
                Summary: "Mentally ill inmates in Miami are housed on the "forgotten floor" Judge Steven Leifman says most are there as a result of "avoidable felonies" While CNN tours facility, patient shouts: "I am the son of the president" Leifman says the system is unjust and he's fighting for change ."
                
                Now summarize the following article:\n\n{article}\n\nFinal Summary:
            """,
    "cot": """
                Summarize the following article step by step:\n\n{article}\n\n
                Step 1: Identify the key points of the article.\n
                Step 2: Combine these key points into a concise summary.\n\n
                Final Summary:
            """,
    "cove": """
                Summarize the following article. After summarizing, verify each sentence against the text:\n\n{article}\n\nFinal Summary:
            """,
    "scot": """
                Summarize the article using the following structure:\n"
                Step 1. Background\n
                Step 2. Key Points\n
                Step 3. Conclusion\n\n
                {article}\n\n
                Final Summary:
            """
}

In [ ]:
def generate_summaries(batch, template, pipeline):
    """
    Generate summaries for a batch of articles using the specified prompt template and pipeline.
    """
    summaries = []
    for article in batch["article"]:
        prompt = template.format(article=article)
        response = pipeline(prompt, max_new_tokens=512, truncation=True, max_length=512)
        
        if isinstance(response, list) and "generated_text" in response[0]:
            generated_text = response[0]["generated_text"]
        elif isinstance(response, list) and "summary_text" in response[0]:
            generated_text = response[0]["summary_text"]
        else:
            generated_text = str(response)

        # Extract summary
        summary_start = generated_text.find("Final Summary:") + len("Final Summary:")
        summary = generated_text[summary_start:].strip()
        summaries.append(summary)
    
    return {"summary": summaries}

In [ ]:
for pipeline_name, pipeline in pipelines.items():
    for strategy, template in templates.items():
        # prompt_template = PromptTemplate(template=template, input_variables=["article"])
        
        dataset = dataset.map(
            lambda batch: generate_summaries(batch, template, pipeline),
            batched=True,
            batch_size=8
        )
        
        test_sample[f"{strategy}_{pipeline_name}_summary"] = dataset["summary"]

In [ ]:
test_sample.to_csv("summarization_llama31_results.csv", index=False, encoding='utf-8')

In [ ]:
test_sample.head()

In [ ]:
# test_sample.article[0]